# Local configuration regression checks

All data here are documentation fixtures. No real instrument connection or upload occurs. Test code is visible in each cell; real identities are never required.

In [ ]:
import json,copy,tempfile
from pathlib import Path
from Software.tools.bench_configuration import read_units,select_unit,render_header
from malab.bench_config import load_bench_config,dashboard_options
fixture={'units':[{'name':'unit1','address':'192.0.2.10','mac':'02:00:00:00:00:01','usb_port':'FAKE1','usb_serial':'SERIAL1','calibration':[None]*4},{'name':'unit2','address':'192.0.2.20','mac':'02:00:00:00:00:02','usb_port':'FAKE2','usb_serial':'SERIAL2','calibration':[None]*4}], 'meter':{'address':'192.0.2.71','channel':5}}
tmp=tempfile.TemporaryDirectory();path=Path(tmp.name)/'bench.local.json'
def write(value):path.write_text(json.dumps(value),encoding='utf-8')
write(fixture)


In [ ]:
# Add a third unit without changing firmware source or channel-map code.
third=copy.deepcopy(fixture)
third['units'].append(dict(name='unit3',address='192.0.2.30',mac='02:00:00:00:00:03',usb_port=None,usb_serial=None,calibration=[None]*4))
write(third);c=read_units(path);u=select_unit(c,'unit3');h=render_header(u)
assert '192,0,2,30' in h and '0x03' in h and h.count('nominalCalibration')==4
options=dashboard_options(path)
assert options['addresses']==['192.0.2.10','192.0.2.20','192.0.2.30']
assert options['bench_ports']==['FAKE1','FAKE2',None]
assert options['meter_channel']==5
print('PASS: third unit generates its own identity and occupies channels 9-12')


In [ ]:
# A measured calibration belongs only to the unit that owns it.
c=copy.deepcopy(fixture)
c['units'][0]['calibration'][0]=dict(offset_nanovolts=-9992845256,step_picovolts=76283553,zero_code=130996,measured=True)
write(c);c=read_units(path)
one=render_header(select_unit(c,'unit1'));two=render_header(select_unit(c,'unit2'))
assert '130996' in one and '130996' not in two
assert one.count('nominalCalibration')==3 and two.count('nominalCalibration')==4
print('PASS: identity and calibration stay paired')


In [ ]:
# Ambiguous or invalid identities must fail before building/connecting.
for key in ['name','address','mac','usb_port','usb_serial']:
 c=copy.deepcopy(fixture);c['units'][1][key]=c['units'][0][key];write(c)
 try:read_units(path)
 except ValueError:pass
 else:raise AssertionError('Accepted duplicate '+key)
for value in ['01:00:00:00:00:01','00:00:00:00:00:00','bad']:
 c=copy.deepcopy(fixture);c['units'][0]['mac']=value;write(c)
 try:read_units(path)
 except ValueError:pass
 else:raise AssertionError('Accepted invalid MAC')
write(fixture)
try:select_unit(read_units(path),'absent')
except ValueError:pass
else:raise AssertionError('Missing unit fell back to another box')
try:load_bench_config(path.with_name('missing.json'))
except ValueError:pass
else:raise AssertionError('Missing file fell back to a real address')
print('PASS: duplicates, invalid MACs, missing units/files fail closed')


In [ ]:
# Dashboard rejects a meter on a nonexistent channel and duplicate USB mappings.
for channel in [0,9,True]:
 c=copy.deepcopy(fixture);c['meter']['channel']=channel;write(c)
 try:load_bench_config(path)
 except ValueError:pass
 else:raise AssertionError('Accepted invalid meter channel')
c=copy.deepcopy(fixture);c['units'][1]['usb_port']='fake1';write(c)
try:load_bench_config(path)
except ValueError:pass
else:raise AssertionError('Accepted duplicate USB port with different case')
tmp.cleanup()
print('PASS: dashboard association validation')
